In [2]:
import os
import time
import json
import requests
import pandas as pd
import lyricsgenius
from openai import OpenAI
from typing import Optional, Tuple
from tqdm import tqdm
import numpy as np
import re
import base64
import ast

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

tqdm.pandas(desc="fetching lyrics")

In [7]:
df_tophit = pd.read_csv("tophit_charts/tophit_master_all.csv")
df_pg = pd.read_csv("song_of_the_year_charts/song_of_the_year_master_not_cleared.csv")

In [8]:
df = pd.concat([df_tophit, df_pg], ignore_index=True)
df.columns = df.columns.str.lower()
df.shape

(8740, 6)

In [9]:
df = df.drop_duplicates(subset=['artist_name', 'song_name'])
df.shape

(6275, 6)

In [10]:
df = df[
    ~df['artist_name'].astype(str).str.contains('unknown', case=False, na=False) & 
    ~df['song_name'].astype(str).str.contains('unknown', case=False, na=False) &
    df['language'].astype(str).str.contains('russian', case=False, na=False)
].reset_index(drop=True)

df.shape

(4188, 6)

In [ ]:
output_path = "pre_lyrics/final_api_input.csv"
df.to_csv(output_path, index=False, encoding='utf-8')

In [ ]:
ai_client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY", "sk-4c355f2d8d8c4885a616da01b419ad53"), 
    base_url="https://api.deepseek.com"
)
genius = lyricsgenius.Genius(
    os.getenv("GENIUS_API_KEY", "JNKpGBtCKFm_E6_idil9o-Ew-rfzMDiQyPET8Iz9-ZXbdwa-GjcR7ScekF0WQmls"), 
    timeout=15, retries=3, verbose=False
)
genius.remove_section_headers = True

### call 1: get lyrics

In [ ]:
def fetch_lrclib(artist: str, song: str) -> Tuple[Optional[str], Optional[str], Optional[int]]:
    """Fallback to LRCLIB API. Returns (lyrics, source, id)."""
    try:
        res = requests.get(
            "https://lrclib.net/api/get", 
            params={"artist_name": artist, "track_name": song}, 
            timeout=5
        )
        if res.status_code == 200:
            data = res.json()
            return data.get("plainLyrics"), "lrclib", data.get("id")
    except requests.RequestException:
        pass
    return None, None, None

def get_lyrics(artist: str, song: str) -> Tuple[Optional[str], Optional[str], Optional[int]]:
    """Attempt Genius fetch, falling back to LRCLIB. Returns (lyrics, source, id)."""
    try:
        if song_data := genius.search_song(title=song, artist=artist):
            if song_data.lyrics:
                return song_data.lyrics, "genius", song_data.id
    except Exception:
        pass
    
    return fetch_lrclib(artist, song)

def get_ai_aliases(artist: str, song: str) -> list[Tuple[str, str]]:
    prompt = (
        f"Raw Artist: '{artist}', Raw Song: '{song}'.\n"
        "The input data is often messy. Fix it to determine the true core artist and song name compatible with lyrics aggregators (Genius, Musixmatch).\n"
        "Resolve common issues such as:\n"
        "- Swapped artist and song names.\n"
        "- Extraneous quotes/brackets («») or metadata (e.g., 'Песня-82', years).\n"
        "- Multiple artists (extract ONLY the primary one).\n"
        "Generate up to 3 clean search combinations, including English aliases/transliterations (e.g., 'Уматурман' -> 'Uma2rman').\n"
        "Return ONLY JSON: {\"combinations\": [[\"Artist\", \"Song\"]]}"
    )
    try:
        resp = ai_client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        return json.loads(resp.choices[0].message.content).get('combinations', [])
    except Exception as e:
        print(f"AI Error for {artist}: {e}")
        return []

def process_track(row: pd.Series) -> pd.Series:
    orig_art, orig_sng = str(row['artist_name']), str(row['song_name'])
    if orig_art == 'nan' or orig_sng == 'nan':
        return pd.Series([None, None, None, "not found", None, None])

    # 1. fast path
    lyrics, source, source_id = get_lyrics(orig_art, orig_sng)
    if lyrics:
        return pd.Series([orig_art, orig_sng, "[]", lyrics, source, source_id])

    # 2. ai path
    aliases = get_ai_aliases(orig_art, orig_sng)
    aliases_str = json.dumps(aliases, ensure_ascii=False) 

    for alt_art, alt_sng in aliases:
        time.sleep(0.5)
        lyrics, source, source_id = get_lyrics(alt_art, alt_sng)
        if lyrics:
            return pd.Series([alt_art, alt_sng, aliases_str, lyrics, source, source_id])

    return pd.Series([None, None, aliases_str, "not found", None, None])


df.columns = df.columns.str.lower()

In [ ]:
new_cols = ['artist_matched', 'song_matched', 'ai_suggestions', 'lyrics', 'source', 'source_id']
df[new_cols] = df.progress_apply(process_track, axis=1, result_type='expand')
df.to_csv('output_with_lyrics.csv', index=False, encoding='utf-8')

fetching lyrics:   0%|          | 0/4188 [00:00<?, ?it/s]

fetching lyrics: 100%|██████████| 4188/4188 [12:28:51<00:00, 10.73s/it]    


In [15]:
df.assign(lyrics_status=np.where(df['lyrics'] == 'not found', 'not found', 'filled')).groupby(['year', 'lyrics_status']).size().reset_index(name='count')

,year,lyrics_status,count
0,1991,filled,15
1,1991,not found,32
2,1993,filled,13
3,1993,not found,32
4,1994,filled,8
5,1994,not found,26
6,1995,filled,27
7,1995,not found,14
8,1996,filled,26
9,1996,not found,23


In [16]:
df.sample(3)

,year,platform,artist_name,song_name,release_date,language,artist_matched,song_matched,ai_suggestions,lyrics,source,source_id
1383,2016,radio,Дима Билан,Неделимые,2015-12-02,russian,Дима Билан,Неделимые,[],"5 ContributorsНеделимые (Indivisible) Lyrics\n\nУслышав радостную весть\nЛюбовь хотела новизны\nТвой образ заново воскрес\nХочу гротеска и весны\nТвоё коралловое платье\nИ длинный шлейф забытых дней\nВедут любовь мою к соблазну\nИ наша страсть ещё сильней\nИ я беру тебя за руку\nПростив, за что простить не мог\nКидая на спину разлуку\nДаю твоей любви урок\n\nМы давно с тобою стали неделимые\nНочи рваные, длинные\nНаши чувства до безумия ранимые -\nНочи длинные, длинные\nМы давно с тобою стали неделимые\nНочи рваные, длинные\nНаши чувства до безумия ранимые -\nНочи длинные, длинные\nОпять\n\nПрошу тебя, не прекословь\nНе говори, что ерунда\nЯ верю: первая любовь\nРазрушить cможет города\nВернет забытые закаты\nУставший день, где я увяз\nЗелёный взгляд твой виноватый\nИ поцелуй, не торопясь\nИ я беру тебя за руку\nПростив, за что простить не мог\nКидая на спину разлуку\nДаю твоей любви урок\n\nМы давно с тобою стали неделимые\nНочи рваные, длинные\nНаши чувства до безумия ранимые -\nНочи длинные, длинные\nМы давно с тобою стали неделимые\nНочи рваные, длинные\nНаши чувства до безумия ранимые -\nНочи длинные, длинные\nОпять\n\nМы давно с тобою стали неделимые\nМы давно с тобою стали неделимые\nНеделимые…\n\nМы давно с тобою стали...\nНочи рваные, длинные\nНаши чувства до безумия ранимые -\nНочи длинные, длинные\nМы давно с тобою стали неделимые\nНочи рваные, длинные\nНаши чувства до безумия ранимые -\nНочи длинные, длинные\nОпять",genius,5023211.0
2116,2025,yandex,Voskresenskii,Быть богатым,NaN,russian,Voskresenskii,Быть богатым,[],"28 ContributorsБыть богатым (Be Rich) Lyrics\n\n(Пошёл) Пошёл на хуй, броук на бедном (Ха)\nБыть богатым охуенно\nУ меня дорогое время (Я)\nБыть богатым охуенно (Ха)\nЯ есть современность (Ха, да)\nБыть богатым охуенно\nЯ припоминаю бедность (Bih')\nНо быть богатым охуенней\n\nЩас не покупаю хату (Ири), нахуй подстраховку\nЯ всегда буду на хайпе (Всегда буду на хайпе)\nЯ всегда буду богатым (Бр)\nМоя сука тоже, если мне будет приятно (Ха)\nДаже если я ей изменю, она простит (Простит)\nУ меня тоже есть свэг, и у меня есть стиль (Да)\nУчит мои треки, да, она любит тусить (Ага)\nПро мою жизнь можно снять охуеннейший фильм (По-по)\nЯ не могу понять, как жил без флекса (Nah, ха)\nТрачу рубли на то, на то и это (Тратим всё)\nЗнаешь ли ты, где я был прошлым летом? (Ха, ха)\n\nПомню, когда я был бедным, а, мне было так галимо\nУ тебя нету котлеты, а, с тобой чиллит бомжиха\nЯ люблю, как пахнут деньги, эй, пизже всякого парфюма\nМалая дрочит мои яйца, а, малая хочет сыра\nНенавижу грязных сучек, да\nЯ люблю лишь чистых, не вонючих\nНенавидел ходить на работу, а\nЯ знал, что это ненадолго, я всегда буду везучим\nЛюбит, как я смотрю ей на жопу, а\nЕсли бы я был на лейме, думала, я озабочен\nОна приехала ко мне — ебёмся, эй\nЕё парень на броуке, ей вдруг стало одиноко, а\n\nЯ люблю деньги и шалав\nОна хочет почиллить с нами, но не знает, как сказать, а\nДа, весь этот свэг не просто так\nОна врубает, что тут много, но не может сосчитать, а\n\nПошёл на хуй, броук на бедном (Ха)\nБыть богатым охуенно\nУ меня дорогое время (Я)\nБыть богатым охуенно (Ха)\nЯ есть современность (Ха, да)\nБыть богатым охуенно\nЯ припоминаю бедность (Bih')\nНо быть богатым охуенней",genius,12755732.0
1662,2020,radio,Dabro,Юность,2020-07-21,russian,Dabro,Юность,[],"8 ContributorsЮность (Youth) LyricsЗаглавная песня с нового альбома группы Dabro – “Юность”. Выход альбома состоялся 7 ноября 2020 года.\n\nЗвук поставим на всю и соседи не спят\nКто под нами внизу, вы простите меня\nА потом о любви говорить до утра\nЭто юность моя, это юность моя\nЗвук поставим на всю и соседи не спят\nКто под нами внизу, вы простите меня\nА потом о любви говорить до утра\nЭто юность моя, это юность моя\n\nЗнаю, мы сегодня точно не уснём\nЗнаю, будем до утра

### call 2: get brands

In [20]:
df_call2 = df[df['lyrics'] != 'not found']
print(df_call2.shape)

(3304, 12)


In [ ]:
tqdm.pandas(desc="Extracting Brands")

def extract_brands_from_lyrics(lyrics: str) -> str:
    """Uses DeepSeek to extract brand entities from lyrics, handling slang and returning JSON."""
    if not isinstance(lyrics, str) or not lyrics.strip():
        return "[]"

    prompt = (
        "You are an expert linguistic analyst. Extract all brand names, companies, designers, "
        "car models, and tech products mentioned in the provided lyrics. \n\n"
        "CRITICAL RULES:\n"
        "1. Identify official names, transliterations, abbreviations, and street slang.\n"
        "2. Examples of targets:\n"
        "   - Cars: гелик/мерс (Mercedes-Benz), бэха/бумер (BMW), порш/каен (Porsche), ваз/тазик (LADA), роллс (Rolls-Royce).\n"
        "   - Fashion: гучи/gucci (Gucci), луи/луи витон (Louis Vuitton), стоник (Stone Island), прада (Prada), баленсиага (Balenciaga).\n"
        "   - Tech/Other: айфон (Apple/iPhone), мак (MacBook/Apple), макдак (McDonald's), спрайт (Sprite), ролекс (Rolex).\n"
        "3. Ignore song metadata (e.g., 'Contributors', 'Lyrics', '[Verse 1]').\n"
        "4. If no brands are found, return an empty list.\n\n"
        f"LYRICS:\n{lyrics}\n\n"
        "Return ONLY JSON in this format: {\"brands\": [{\"mention\": \"exact word in text\", \"normalized\": \"Official Brand Name\"}]}"
    )

    try:
        resp = ai_client.chat.completions.create(
            model="deepseek-chat",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            response_format={"type": "json_object"}
        )
        brands_list = json.loads(resp.choices[0].message.content).get('brands', [])
        return json.dumps(brands_list, ensure_ascii=False)
    except Exception as e:
        print(f"Extraction Error: {e}")
        return "[]"


df_call2 = df_call2.copy()

In [ ]:
df_call2['extracted_brands'] = df_call2['lyrics'].progress_apply(extract_brands_from_lyrics)
df_call2.to_csv('output_with_lyrics_and_brands.csv', index=False, encoding='utf-8')

Extracting Brands: 100%|██████████| 3304/3304 [1:38:12<00:00,  1.78s/it]    


In [31]:
df_call2.sample(20)

,year,platform,artist_name,song_name,release_date,language,artist_matched,song_matched,ai_suggestions,lyrics,source,source_id,extracted_brands
1407,2016,radio,Monatik,Кружит,2016-06-24,russian,Monatik,Кружит,[],"9 ContributorsКружит (Revolves) Lyrics\n\nТак особенно...\nТак особенно...\n\nТак особенно сегодня чувства кипят\nТак говорят, а я не вижу их\nТак особенно сегодня люди шалят\nТак говорят, а я не вижу их\nТак особенно сегодня звезды горят\nТак говорят, а я не вижу их ведь\nЗанят мой взгляд...\n\nЭндорфины пошли, фантазия пошлит\nЯ что ли чувствами залит, что глаза мозолю\nЭндорфины пошли, фантазия пошлит\nЭй, ты что ли золотом лита?\nЧто глаза мозолю?\n\nУ-у-у, оказалось, вокруг тебя весь мир кружит!\nВокруг тебя весь мир кружит!\nВокруг тебя весь мир кружит!\nКружит, кружит... о-у, как жить?\n\nКружит голову, ай до упаду\nГолову, да, до упаду\nКружит голову, ай до упаду\nГолову, да... (дай мне!)\nГоловы кружит, головы кружит...\n\nТак особенно сегодня чувства кипят\nТак говорят, а я не вижу их\nТак особенно сегодня люди шалят\nТак говорят, а я не вижу их\nТак особенно сегодня звезды горят\nТак говорят, а я не вижу их\nЗанят мой взгляд...\n\nЭндорфины пошли, фантазия пошлит\nЯ что ли чувствами залит, что глаза мозолю\nЭндорфины пошли, фантазия пошлит\nЭй, ты что ли золотом лита?\nЧто глаза мозолю?\n\nУ-у-у, оказалось, вокруг тебя весь мир кружит!\nВокруг тебя весь мир кружит!\nВокруг тебя весь мир кружит!\nКружит, кружит... о-у, как жить?\n\nКружит голову, ай до упаду\nГолову, да, до упаду\nКружит голову, ай до упаду\nГолову, да... (дай мне!)\nГоловы кружит, головы кружит...\n\nТы так особенна, ты так особенна\nВокруг тебя весь мир кружит!\nПо тебе с ума сходят\nТы так особенна, ты так особенна\nВокруг тебя весь мир кружит!\nУ-у-у-у-у-у-у-у...\n\nОказалось, что...\nОказалось так, что...\nВокруг тебя весь мир\nВокруг тебя весь мир\nВокруг тебя весь мир\nВокруг тебя весь мир\nВокруг тебя весь мир\nВокруг тебя весь мир\n\nВокруг тебя весь мир кружит!\nКружит голову, ай до упаду\nГолову, да, до упаду\nКружит голову, ай до упаду\nГолову, да... (дай мне!)",genius,3114047.0,[]
2110,2025,yandex,Борик,Тыцын-Дыцын,NaN,russian,Борик,Тыцын-Дыцын,[],"12 ContributorsТЫЦН-ДЫЦН (TYTSN-DYTSN) LyricsЭнергичный рэп-трек от Ильи Сатира, выпущенный под псевдонимом БОРИК. Песня была создана специально для видео на популярном ютуб-канале ExileShow, в котором нужно было угадать блогера.\nВ треке Борик читает о… Read More \n\nБорик!\nБорик!\n\nЛето пролетело, словно параплан\nТело загорело, парам-пара-рам\nСладкая конфета, тает по ночам\nХуди цвета кирпича, мы танцуем до утра\n\nИ ты пришла сюда в короткой юбке\nИ ты сошла с ума на третьи сутки\nИ эта девочка моя — не шутки\nВесь город знает, что…\n\nЯ тебя тыцын-дыцын\nЯ тебя тыцын-дыцын\nЯ тебя тыцын-дыцын\nЯ тебя тыцын-дыцын\nЯ тебя…\n\nТыц-тыдыц-тыдыц-тыц-тыц-тыц\nТы-ды-ды-ды-ды-дыц-тыц-тыц\nЯ тебя тыц-тыц, я тебя дыц-дыц\nТы-ды-ды-тыц, тыц-ды-ды-дыц\nТыц-ты-дыц, тыц-ты-дыц-дыц\nТы-ды-ды-ды, ты-ды-ты-ды-тыц\nТыц-тыц, тыц, ты-ты-ды-тыц\nЯ тебя ты-цу-дыц, тыц-тыц-тудыц",genius,12473792.0,[]
1007,2010,radio,Вєрка Сердючка,Дольче Габбана,2010-08-06,russian,Вєрка Сердючка,Дольче Габбана,[],"6 ContributorsДольче Габбана (Dolce Gabbana) Lyrics\n\nА я иду такая вся\nВ Дольче Габбана\nЯ иду такая вся\nНа сердце рана\nСлёзы душат-душат\nЯ в плену обману\nНо иду такая вся\nВ Дольче Габбана\nЛалала-уах-уах-лалала\nЛалала-уах-уах-лалала\nЛалала-уах-уах-лалала\nЛалала-уах-уах-лалала\n\nМне уже восемнадцать\nВ паспорт страшно смотреть\nОн сказал, я красивее\nВсех на планете\nПочему ночью злой\nНе смогла сказать ънетъ\nДайте мне вина, у-а\nНу дайте пачку сигарет\nПочему в были вместе\nТы мне объясни\nУ него КиевСтар\nУ меня ЮМС\nЧто такое любовь\nКто откроет секрет?\nДайте мне вина, у-а\nНу дайте пачку сигарет\n\nА я иду такая вся (вся)\nВ Дольче Габбана\nЯ иду такая вся (вся)\nНа сердце рана\nСлёзы душат-душат\nГолос мой простужен\nЯ иду такая вся (вся)\nА он мне не нужен\nА я иду такая вся (вся)\nВ Доль

### cell 3: metadata addition

In [14]:
df = pd.read_csv('pre_metadata/output_with_lyrics_and_brands.csv')

In [23]:
df['lyrics'] = df['lyrics'].str.replace(
    r'^\d+\s*Contributors?.*?Lyrics\s*', 
    '', 
    regex=True, 
    flags=re.IGNORECASE
)

In [ ]:
LASTFM_API_KEY = "e230cd7998903ddb5876beb486e82ee2"

In [69]:
def get_itunes_genres(artist: str) -> list[str]:
    res = requests.get(
        "https://itunes.apple.com/search", 
        params={"term": artist, "entity": "musicArtist", "limit": 1}
    ).json()
    results = res.get("results", [])
    return [results[0].get("primaryGenreName")] if results and "primaryGenreName" in results[0] else []

def get_yandex_genres(artist: str) -> list[str]:
    res = requests.get(
        "https://api.music.yandex.net/search", 
        params={"text": artist, "type": "artist", "page": 0},
        headers={"X-Yandex-Music-Client": "YandexMusicAndroid/24023231"}
    )
    
    if res.status_code != 200:
        return []
        
    artists = res.json().get("result", {}).get("artists", {}).get("results", [])
    
    if artists and "genres" in artists[0]:
        genres = artists[0].get("genres", [])
        return [g.get("title", g) if isinstance(g, dict) else g for g in genres]
    return []

def get_lastfm_genres(artist: str, api_key: str, limit: int = 5) -> list[str]:
    res = requests.get(
        "https://ws.audioscrobbler.com/2.0/", 
        params={"method": "artist.gettoptags", "artist": artist, "api_key": api_key, "format": "json"}
    ).json()
    tags = res.get("toptags", {}).get("tag", [])
    tags = [tags] if isinstance(tags, dict) else tags 
    return [tag.get("name") for tag in tags[:limit]]

def get_musicbrainz_genres(artist: str) -> list[str]:
    res = requests.get(
        "https://musicbrainz.org/ws/2/artist", 
        headers={"User-Agent": "DataPipeline/1.0 ( your@email.com )"}, 
        params={"query": artist, "limit": 1, "fmt": "json"}
    ).json()
    artists = res.get("artists", [])
    return [tag.get("name") for tag in artists[0].get("tags", [])] if artists else []

In [ ]:
def enrich_all_source_genres(df: pd.DataFrame, lastfm_key: str) -> pd.DataFrame:
    unique_artists = df['artist_matched'].dropna().unique()
    
    itunes_map, yandex_map, lastfm_map, mb_map = {}, {}, {}, {}

    for artist in tqdm(unique_artists, desc="Fetching Genres"):
        # 1. itunes
        try:
            itunes_map[artist] = get_itunes_genres(artist)
        except Exception as e:
            itunes_map[artist] = []

        # 2. ymusic
        try:
            yandex_map[artist] = get_yandex_genres(artist)
        except Exception as e:
            yandex_map[artist] = []

        # 3. lastfm
        try:
            lastfm_map[artist] = get_lastfm_genres(artist, lastfm_key)
        except Exception as e:
            lastfm_map[artist] = []

        # 4. musicbrainz
        try:
            mb_map[artist] = get_musicbrainz_genres(artist)
        except Exception as e:
            mb_map[artist] = []
            
        time.sleep(1) 

    df['genres_itunes'] = df['artist_matched'].map(itunes_map)
    df['genres_yandex'] = df['artist_matched'].map(yandex_map)
    df['genres_lastfm'] = df['artist_matched'].map(lastfm_map)
    df['genres_musicbrainz'] = df['artist_matched'].map(mb_map)
    
    return df

In [ ]:
enriched_df = enrich_all_source_genres(
    df=df.copy(),
    lastfm_key=LASTFM_API_KEY
)

Fetching Genres: 100%|██████████| 965/965 [46:51<00:00,  2.91s/it]


In [ ]:
enriched_df['genres_yandex'].astype(str).value_counts()

genres_yandex
['ruspop']                                       1224
['rusestrada']                                    775
['rusrap']                                        266
['rusrock']                                       142
['ruspop', 'dance', 'pop']                         75
['ruspop', 'rusrap', 'pop']                        45
['rusrap', 'ruspop', 'rap']                        44
['caucasian']                                      43
['ruspop', 'rusrap', 'dance']                      40
['pop']                                            34
['rusrap', 'rap']                                  29
['local-indie']                                    25
['estrada']                                        25
['ruspop', 'pop', 'rusrap']                        23
['ruspop', 'pop']                                  21
['shanson']                                        20
['rusrap', 'dance', 'ruspop']                      19
[]                                                 19
['ruspop', 'po

In [85]:
df_genres = (
    enriched_df.assign(
        # coalesce
        year=lambda df: df['release_date'].fillna(df['year']),
        
        # case when
        genius_id=lambda df: np.where(df['source'] == 'genius', df['source_id'], np.nan),
        lrclib_id=lambda df: np.where(df['source'] == 'lrclib', df['source_id'], np.nan)
    )
    .rename(columns={
        'artist_name': 'raw_artist',
        'song_name': 'raw_song',
        'artist_matched': 'matched_artist',
        'song_matched': 'matched_song'
    })
    [[
        'year', 'platform', 'raw_artist', 'raw_song', 'matched_artist', 
        'matched_song', 'ai_suggestions', 'lyrics', 'genius_id', 'lrclib_id', 
        'genres_yandex', 'genres_musicbrainz', 'genres_itunes', 'genres_lastfm', 
        'chosen_genre', 'extracted_brands'
    ]]
)

In [4]:
df_genres = pd.read_csv('/Users/vasilii/Documents/python/study/masters/flask/dashboard_data/df_genres.csv')

In [ ]:
GENRE_MAP = {
    'folk': 'folk',
    'electronics': 'electronics',
    'caucasian': 'caucasian',
    'rusrock': 'rock',
    'rock': 'rock',
    'dance': 'dance',
    'rusrap': 'rap',
    'rap': 'rap',
    'rusestrada': 'estrada',
    'estrada': 'estrada',
    'ruspop': 'pop',
    'pop': 'pop'
}

PRIORITY = [
    'folk',         
    'electronics',  
    'caucasian',    
    'rock',         
    'dance',        
    'rap',          
    'estrada',      
    'pop'           
]

def pick_best_genre(genres: list) -> str:
    if not isinstance(genres, list) or not genres:
        return 'other'
        
    mapped_genres = [GENRE_MAP[g] for g in genres if g in GENRE_MAP]
    
    if not mapped_genres:
        return 'other'
        
    return min(mapped_genres, key=lambda g: PRIORITY.index(g))

In [16]:
df_genres['chosen_genre'] = df_genres['genres_yandex'].apply(pick_best_genre)

In [18]:
df_genres.chosen_genre.value_counts()

chosen_genre
pop            1327
estrada         820
rap             521
dance           207
rock            189
other           114
caucasian        56
electronics      39
folk             31
Name: count, dtype: int64

In [20]:
df_genres.to_csv('/Users/vasilii/Documents/python/study/masters/flask/pre_draw/df_genres.csv', index=False, encoding='utf-8')